<a href="https://colab.research.google.com/github/massengineer/FYP-ML-Bird-Scarer/blob/main/FYP_YOLO_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Verify NVIDIA GPU Availability**

Make sure you're using a GPU-equipped machine by going to "Runtime" -> "Change runtime type" in the top menu bar, and then selecting one of the GPU options in the Hardware accelerator section. Click Play on the following code block to verify that the NVIDIA GPU is present and ready for training.

In [1]:
!nvidia-smi

Mon Feb 23 22:05:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#1.&nbsp;Gather and Label Training Images

# 2.&nbsp;Upload Image Dataset and Prepare Training Data

## 2.1 Upload images

First, we need to upload the dataset to Colab. Here are a few options for moving the `data.zip` folder into this Colab instance.


**Option 2. Copy from Google Drive**

You can also upload your images to your personal Google Drive, mount the drive on this Colab session, and copy them over to the Colab filesystem. This option works well if you want to upload the images beforehand so you don't have to wait for them to upload each time you restart this Colab. If you have more than 50MB worth of images, I recommend using this option.

First, upload the `data.zip` file to your Google Drive, and make note of the folder you uploaded them to. Replace `MyDrive/path/to/data.zip` with the path to your zip file. (For example, I uploaded the zip file to folder called "candy-dataset1", so I would use `MyDrive/candy-dataset1/data.zip` for the path). Then, run the following block of code to mount your Google Drive to this Colab session and copy the folder to this filesystem.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

!cp /content/gdrive/MyDrive/path/to/data.zip /content

## 2.2 Split images into train and validation folders

In [2]:
# Unzip images to a custom data folder
!unzip -q /content/data.zip -d /content/custom_data


I wrote a Python script that will automatically create the required folder structure and randomly move 90% of dataset to the "train" folder and 10% to the "validation" folder. Run the following code block to download and execute the scrpt.

In [3]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

--2026-02-23 22:09:15--  https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3203 (3.1K) [text/plain]
Saving to: ‘/content/train_val_split.py’

/content/train_val_ 100%[===================>]   3.13K  --.-KB/s    in 0s      

2026-02-23 22:09:15 (65.3 MB/s) - ‘/content/train_val_split.py’ saved [3203/3203]

Created folder at /content/data/train/images.
Created folder at /content/data/train/labels.
Created folder at /content/data/validation/images.
Created folder at /content/data/validation/labels.
Number of image files: 153
Number of annotation files: 153
Images moving to train: 137
Images moving to validation: 16


# 3.&nbsp;Install Requirements (Ultralytics)

In [4]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.6 MB/s eta 0:00:00


# 4.&nbsp;Configure Training


In [5]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

Created config file at /content/data.yaml

File contents:

path: /content/data
train: train/images
val: validation/images
nc: 6
names:
- duck
- pheasant
- pigeon
- raven
- sparrow
- starling


# 5.&nbsp;Train Model

## 5.2 Run Training

In [9]:
!yolo detect train data=/content/data.yaml model=yolov8n.pt epochs=200 imgsz=640

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

#6.&nbsp;Test Model

In [10]:
!yolo detect predict model=runs/detect/train/weights/best.pt source=data/validation/images save=True

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

image 1/16 /content/data/validation/images/03e757e0-common_raven_3_28.jpg: 64x64 1 raven, 9.4ms
image 2/16 /content/data/validation/images/0753e562-house_sparrow_1_44.jpg: 64x64 1 sparrow, 5.7ms
image 3/16 /content/data/validation/images/14554937-65615291-480px.jpg: 64x64 1 starling, 5.7ms
image 4/16 /content/data/validation/images/2ab542b1-european_starling_62_3.jpg: 64x64 1 starling, 5.3ms
image 5/16 /content/data/validation/images/3b897cd5-house_sparrow_2_16.jpg: 64x64 1 starling, 5.5ms
image 6/16 /content/data/validation/images/3bd96574-common_raven_2_25.jpg: 64x64 1 raven, 5.1ms
image 7/16 /content/data/validation/images/4b6e5ebb-SSPCA-Shoot_05_grey.jpg: 64x64 1 starling, 5.1ms
image 8/16 /content/data/validation/images/5b0f30b8-european_starling_64_23.jpg: 64x64 2 starlings, 5.1ms
image 9/16 /content/data/validation/im

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:20]:
  display(Image(filename=image_path, height=400))
  print('\n')


#7.&nbsp;Deploy Model

## 7.1 Download YOLO Model

In [13]:
# Create "my_first_trained_model" folder to store model weights and train results
!mkdir /content/my_first_trained_model
!cp /content/runs/detect/train/weights/best.pt /content/my_first_trained_model/my_first_trained_model.pt
!cp -r /content/runs/detect/train /content/my_first_trained_model

# Zip into "my_model.zip"
%cd my_first_trained_model
!zip /content/my_first_trained_model.zip my_first_trained_model.pt
!zip -r /content/my_first_trained_model.zip train
%cd /content

/content/my_first_trained_model
  adding: my_first_trained_model.pt (deflated 9%)
  adding: train/ (stored 0%)
  adding: train/train_batch1.jpg (deflated 1%)
  adding: train/results.png (deflated 6%)
  adding: train/confusion_matrix_normalized.png (deflated 26%)
  adding: train/train_batch450.jpg (deflated 4%)
  adding: train/results.csv (deflated 60%)
  adding: train/labels.jpg (deflated 32%)
  adding: train/val_batch0_pred.jpg (deflated 20%)
  adding: train/train_batch452.jpg (deflated 3%)
  adding: train/BoxF1_curve.png (deflated 9%)
  adding: train/train_batch0.jpg (deflated 1%)
  adding: train/BoxPR_curve.png (deflated 17%)
  adding: train/val_batch0_labels.jpg (deflated 19%)
  adding: train/weights/ (stored 0%)
  adding: train/weights/last.pt (deflated 9%)
  adding: train/weights/best.pt (deflated 9%)
  adding: train/BoxP_curve.png (deflated 11%)
  adding: train/BoxR_curve.png (deflated 12%)
  adding: train/train_batch451.jpg (deflated 3%)
  adding: train/args.yaml (deflated 53%)

In [14]:
# This takes forever for some reason, you can also just download the model from the sidebar
from google.colab import files

files.download('/content/my_first_trained_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>